In [1]:
import numpy as np
from time import time
from HARK.models import HabitPortfolioConsumerType, RiskyAssetConsumerType
from HARK.ConsumptionSaving.ConsHabitModel import (
    HabitPortfolioConsumerType_defaults,
)

mystr = lambda x: "{:.3f}".format(x)

In [2]:
# Make a parameter dictionary
my_params = HabitPortfolioConsumerType_defaults.copy()
del my_params["constructors"]  # don't want to overwrite these
adjusted_params = {
    "CRRA": 3.5,
    "LivPrb": [1.0],
    "DiscFac": 0.94,
    "PermGroFac": [1.00],
    "Rfree": [1.01],
    "RiskyAvg": 1.04,
    "RiskyStd": 0.18,
    "RiskyShareFixed": None,
    "hLogInitMean": 0.0,
    "HabitMax": 8.0,
    "HabitCount": 51,
    "cycles": 0,
}
my_params.update(adjusted_params)

In [3]:
# Define grid specifications
wealth_grid = {"min": 0.0, "max": 120.0, "N": 201, "order": 2.5}
con_grid = {"min": 0.0, "max": 3.0, "N": 301, "order": 1.1}
share_grid = {"min": 0.0, "max": 1.0, "N": 101}
habit_grid = {"min": 0.3, "max": 3.0, "N": 76, "order": 2.0}
my_grids_base = {
    "kNrm": wealth_grid,
    "wNrm": wealth_grid,
    "qNrm": wealth_grid,
    "cNrm": con_grid,
    "Share": share_grid,
}
my_grids = my_grids_base.copy()
my_grids["hPre"] = habit_grid
my_grids["hNrm"] = habit_grid

In [4]:
# Make and solve the base type with no habits
BaseType = RiskyAssetConsumerType(**my_params)
t0 = time()
BaseType.solve()
BaseType.initialize_sym()
BaseType._simulator.make_transition_matrices(my_grids_base, norm="PermShk")
BaseType._simulator.find_steady_state()
t1 = time()
print("Solving the model with no habits took " + mystr(t1 - t0) + " seconds.")

Solving the model with no habits took 3.293 seconds.


In [5]:
# Calculate target assets and risky share
a_targ = BaseType._simulator.get_long_run_average("aNrm")
w_targ = BaseType._simulator.get_long_run_average("wNrm")
q_targ = BaseType._simulator.get_long_run_average("qNrm")
s_targ = q_targ / w_targ
print(a_targ, s_targ)

12.552368247847019 0.6945006925356999


In [6]:
# Define a function that returns the weighted distance of long run averages
def calc_distance(alpha, beta, lamda, rho):
    temp_params = my_params.copy()
    temp_params["HabitWgt"] = alpha
    temp_params["DiscFac"] = beta
    temp_params["HabitRte"] = lamda
    temp_params["CRRA"] = rho
    TempType = HabitPortfolioConsumerType(**temp_params)

    TempType.solve()
    TempType.initialize_sym()
    TempType._simulator.make_transition_matrices(my_grids, norm="PermShk")
    TempType._simulator.find_steady_state()
    a_val = TempType._simulator.get_long_run_average("aNrm")
    w_val = TempType._simulator.get_long_run_average("wNrm")
    q_val = TempType._simulator.get_long_run_average("qNrm")
    s_val = q_val / w_val

    distance = np.sqrt((a_val - a_targ) ** 2 + (10 * (s_val - s_targ)) ** 2)
    print(alpha, beta, lamda, rho, a_val, s_val)
    return distance

In [7]:
# Define a temporary function to minimize
def my_func(x):
    varphi = x[0]
    rho = x[1]
    beta = 1.0 / (1.0 + np.exp(varphi))
    return calc_distance(0.8, beta, 0.1, rho)

In [8]:
# Minimize the distance
# out = minimize_nelder_mead(my_func, [-2.7, 5.9])

In [9]:
1.0 / (1.0 + np.exp(-2.55))

np.float64(0.9275735146384823)